In [4]:
#importing the tiny philosopher dataset
# from pathlib import Path

# project_dir = Path(__file__).absolute().parent.parent
# dataset_path = project_dir / "datasets" / "tinyphilosohper.txt"

with open("tinyphilosopher.txt", 'r', encoding = 'utf-8') as f:
    text = f.read()
text[:100]

'the ego builds walls, the soul seeks horizons.\nan inflated ego sees only its own reflection.\ntrue wi'

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

d:\Programs\python (+sql)\Learning_Neural_Networks_From_Scratch


In [3]:
from my_tokenizer.gpt_bpe_tokenizer.serialization import load_tokenizer
from my_tokenizer.gpt_bpe_tokenizer.regex_tokenizer import RegexTokenizer

In [4]:
tokenizer = load_tokenizer(RegexTokenizer, PROJECT_ROOT / "checkpoints" / "tokenizer" / "gpt4tokenizer.model")

In [11]:
ids = torch.tensor(tokenizer.encode(text))

In [12]:
dataset_size = len(ids)
train_size = int(0.9 * dataset_size)
x_train , x_test= ids[:train_size], ids[train_size:]
y_train, y_test = ids[1:train_size+1], ids[train_size+1:]

In [13]:
#B, T shape 
T = 8
seq_len = (len(x_train) // T) * T
seq_len2 = (len(x_test) // T) * T
x_train = x_train[:seq_len].view(-1, T)
y_train = y_train[:seq_len].view(-1, T)
x_test = x_test[:seq_len2].view(-1, T)
y_test = y_test[:seq_len2].view(-1, T)

In [14]:
x_train[:10], y_train[:10]

(tensor([[ 116,  263,  519,  103,  111,  360,  117,  465],
         [ 100,  115,  340,  591,  115,   44,  268,  306],
         [ 309,  108,  306,  608,  115,  331,  258,  638],
         [ 262,  115,  294,  266,  332,  102,  108,  551],
         [ 519,  103,  111,  306,  101,  289,  299,  352],
         [ 941,  316, 1018,  432,  102,  108,  477,  338],
         [ 294,  116,  114,  117,  101,  424,  115,  100],
         [ 288,  360,  702,  784,  523,  601,  268,  519],
         [ 103,  111,   32,  550,  115,  294,  116,  263],
         [ 519,  103,  111,  349,  108,  295,  115,  348]]),
 tensor([[ 263,  519,  103,  111,  360,  117,  465,  100],
         [ 115,  340,  591,  115,   44,  268,  306,  309],
         [ 108,  306,  608,  115,  331,  258,  638,  262],
         [ 115,  294,  266,  332,  102,  108,  551,  519],
         [ 103,  111,  306,  101,  289,  299,  352,  941],
         [ 316, 1018,  432,  102,  108,  477,  338,  294],
         [ 116,  114,  117,  101,  424,  115,  100,  2

In [15]:
x_train.shape, y_train.shape

(torch.Size([142484, 8]), torch.Size([142484, 8]))

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import plotly.graph_objects as go

In [16]:
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [143]:
class RNNCell(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.W_xh = nn.Linear(embedding_dim, hidden_size, bias=True, device=device)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True, device=device)        
        self.W_hy = nn.Linear(hidden_size, vocab_size, bias=True, device=device)
        self.embd = nn.Embedding(vocab_size, embedding_dim, device=device)

    def forward(self, x_t, h_prev):
        h_t = torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))
        y_t = self.W_hy(h_t)
        return y_t, h_t

In [ ]:
vocab_size = 1024
embedding_dim = 42
hidden_size = 150
batch_size = 32
lr = 0.001
steps = 5000 

model = RNNCell(vocab_size, embedding_dim, hidden_size)
optimizer = optim.AdamW(model.parameters(), lr=lr)

In [145]:
def get_loss(split):

    if split == "train":
        model.train()
        x_dataset, y_dataset = x_train, y_train
    else:
        model.eval()
        x_dataset, y_dataset = x_test, y_test
    
    with torch.no_grad():

        row, seq_len = x_dataset.shape
        
        idx = torch.randint(0, row, (batch_size,))
        
        x_batch = x_dataset[idx]
        y_batch = y_dataset[idx]
        
        embds = model.embd(x_batch)

        h_prev = torch.zeros(batch_size, hidden_size, device = device)

        outputs = []

        for t in range(seq_len):

            x_t = embds[:, t, :]
            y_t, h_t = model(x_t, h_prev)
            h_prev = h_t
            outputs.append(y_t)
        
        logits = torch.stack(outputs, dim = 1)

        B, T, C = logits.shape

        loss = F.cross_entropy(logits.reshape(B*T,C), y_batch.reshape(B*T))
        return loss.item()
train_loss = get_loss("train")
val_loss = get_loss("test")
train_loss, val_loss

(6.967657566070557, 7.002608299255371)

In [ ]:
train_losses = []
val_losses = []
for i in range(steps):
    #Forward pass
    #sample batch
    row, T = x_train.shape
    idx = torch.randint(0, row, (batch_size,))

    x_batch = x_train[idx]
    y_batch = y_train[idx]

    #embds
    embds = model.embd(x_batch)
    h_prev = torch.zeros(batch_size, hidden_size, device = device)

    outputs = []

    for t in range(T):

        x_t = embds[:, t, :]
        y_t, h_t = model(x_t, h_prev)
        h_prev = h_t
        outputs.append(y_t)
    
    logits = torch.stack(outputs, dim = 1)

    B, T, C = logits.shape

    loss = F.cross_entropy(logits.reshape(B*T,C), y_batch.reshape(B*T))

    #Backward pass
    optimizer.zero_grad()
    loss.backward()
    #gradient clipping to prevent exploding gradients
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    #Update
    optimizer.step()

    #Print loss
    if i%100 == 0:
        train_loss = get_loss("train")
        val_loss = get_loss("test")
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Step:{i}/{steps} train loss:{train_loss:.4f} test_loss:{val_loss:.4f}")


Epochs:0/5000 train loss:6.9661 test_loss:6.9736
Epochs:100/5000 train loss:4.9859 test_loss:4.9042
Epochs:200/5000 train loss:4.4463 test_loss:4.6136
Epochs:300/5000 train loss:4.1087 test_loss:4.1838
Epochs:400/5000 train loss:3.9606 test_loss:3.9677
Epochs:500/5000 train loss:3.8408 test_loss:3.8374
Epochs:600/5000 train loss:3.4948 test_loss:3.7647
Epochs:700/5000 train loss:3.6117 test_loss:3.6552
Epochs:800/5000 train loss:3.2646 test_loss:3.8153
Epochs:900/5000 train loss:3.3796 test_loss:3.1683
Epochs:1000/5000 train loss:3.2929 test_loss:3.2632
Epochs:1100/5000 train loss:3.1213 test_loss:3.4284
Epochs:1200/5000 train loss:3.0244 test_loss:3.1836
Epochs:1300/5000 train loss:3.2290 test_loss:3.2291
Epochs:1400/5000 train loss:3.0904 test_loss:3.1613
Epochs:1500/5000 train loss:2.9633 test_loss:3.1864
Epochs:1600/5000 train loss:2.9527 test_loss:2.9617
Epochs:1700/5000 train loss:3.1891 test_loss:2.9974
Epochs:1800/5000 train loss:2.9506 test_loss:3.0434
Epochs:1900/5000 train l

In [ ]:
steps = torch.arange(1, len(train_losses)) 
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=train_losses, mode='lines', name='Training Loss', line=dict(width=2)))

fig.add_trace(go.Scatter(x=steps, y=val_losses, mode='lines', name='Validation Loss',line=dict(width=2, dash='dash')))

fig.update_layout(title='RNN Training and Validation Loss',xaxis_title='Steps', yaxis_title='Loss', template='plotly_white', hovermode='x unified')

fig.show()

In [148]:
loss.item()

2.6307225227355957

In [150]:
#Sampling from the model
with torch.no_grad():
    max_tokens = 100
    token = torch.zeros((1, 1), dtype=torch.long, device=device)
    h_prev = torch.zeros(1, hidden_size, device=device)
    generated = []
    for _ in range(max_tokens):
        embds = model.embd(token)
        x_t = embds[:, 0, :]
        y_t, h_t = model(x_t, h_prev)
        h_prev = h_t
        probs = F.softmax(y_t, dim=-1)
        token = torch.multinomial(probs, num_samples=1)
        generated.append(token.item())

    output = tokenizer.decode(generated)
    print(output)

 sacred might.
wer langing remains the mirror.
even meres even all whispers and the face.
proutter, wis commons nown leaves who touch and being care one findates where biopolited by not a raw, or of being butclublic g


In [104]:
#Alternatively we can rather create a RNN language model module/class

In [17]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.W_xh = nn.Linear(embedding_dim, hidden_size, bias=True, device=device)
        self.W_hh = nn.Linear(hidden_size, hidden_size, bias=True, device=device)        
        self.W_hy = nn.Linear(hidden_size, vocab_size, bias=True, device=device)
        self.embd = nn.Embedding(vocab_size, embedding_dim, device=device)
        self.hidden_size = hidden_size

    def forward(self, x):
        batch_size, T = x.shape
        embds = self.embd(x)
        h_prev = torch.zeros(batch_size, self.hidden_size, device = x.device)

        outputs = []

        for t in range(T):

            x_t = embds[:, t, :]
            h_t = torch.tanh(self.W_xh(x_t) + self.W_hh(h_prev))
            y_t = self.W_hy(h_t)
            h_prev = h_t
            outputs.append(y_t)
        
        logits = torch.stack(outputs, dim = 1)
        return logits

In [18]:
vocab_size = 1024
embedding_dim = 42
hidden_size = 150
batch_size = 32
lr = 0.001
epochs = 5000 

model = RNNLanguageModel(vocab_size= vocab_size, embedding_dim= embedding_dim, hidden_size= hidden_size)
optimizer = optim.AdamW(model.parameters(), lr=lr)

In [19]:
#get_loss function simplifies
def get_loss(split):

    if split == "train":
        model.train()
        x_dataset, y_dataset = x_train, y_train
    else:
        model.eval()
        x_dataset, y_dataset = x_test, y_test

    with torch.no_grad():

        # sample batch
        row, _ = x_dataset.shape

        idx = torch.randint(0, row, (batch_size,))

        x_batch = x_dataset[idx]
        y_batch = y_dataset[idx]

        # forward pass
        logits = model(x_batch)

        # loss
        B, T, C = logits.shape

        loss = F.cross_entropy(
            logits.reshape(B * T, C),
            y_batch.reshape(B * T)
        )

    return loss.item()

In [108]:
#Forward pass simplifies

In [20]:
train_losses = []
val_losses = []
for i in range(epochs):
    #Forward pass
    #sample batch
    row, T = x_train.shape
    idx = torch.randint(0, row, (batch_size,))

    x_batch, y_batch = x_train[idx], y_train[idx]
    #embds
    logits = model(x_batch)
    B, T, C = logits.shape
    loss = F.cross_entropy(logits.reshape(B*T,C), y_batch.reshape(B*T))

    #Backward pass
    optimizer.zero_grad()
    loss.backward()
    #gradient clipping to prevent exploding gradients
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    #Update
    optimizer.step()

    #Print loss
    if i%100 == 0:
        train_loss = get_loss("train")
        val_loss = get_loss("test")
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epochs:{i}/{epochs} train loss:{train_loss:.4f} test_loss:{val_loss:.4f}")


Epochs:0/5000 train loss:6.9527 test_loss:6.9747
Epochs:100/5000 train loss:5.0362 test_loss:5.0606
Epochs:200/5000 train loss:4.5649 test_loss:4.5589
Epochs:300/5000 train loss:4.5164 test_loss:4.0099
Epochs:400/5000 train loss:3.8553 test_loss:3.7275
Epochs:500/5000 train loss:3.6549 test_loss:3.8929
Epochs:600/5000 train loss:3.6297 test_loss:3.3945
Epochs:700/5000 train loss:3.4088 test_loss:3.6665
Epochs:800/5000 train loss:3.3563 test_loss:3.2724
Epochs:900/5000 train loss:3.3289 test_loss:3.4392
Epochs:1000/5000 train loss:3.3984 test_loss:3.2358
Epochs:1100/5000 train loss:3.4433 test_loss:3.2299
Epochs:1200/5000 train loss:3.0844 test_loss:3.0238
Epochs:1300/5000 train loss:3.0880 test_loss:3.1294
Epochs:1400/5000 train loss:2.8117 test_loss:3.2174
Epochs:1500/5000 train loss:2.8485 test_loss:3.2010
Epochs:1600/5000 train loss:3.2054 test_loss:3.1450
Epochs:1700/5000 train loss:3.0327 test_loss:3.0112
Epochs:1800/5000 train loss:3.1634 test_loss:2.9222
Epochs:1900/5000 train l

In [21]:
#Plotting the loss
steps = torch.arange(1, len(train_losses)) 
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=train_losses, mode='lines', name='Training Loss', line=dict(width=2)))

fig.add_trace(go.Scatter(x=steps, y=val_losses, mode='lines', name='Validation Loss',line=dict(width=2, dash='dash')))

fig.update_layout(title='RNN Training and Validation Loss',xaxis_title='Steps', yaxis_title='Loss', template='plotly_white', hovermode='x unified')

fig.show()

In [23]:
loss.item()

2.5823476314544678

In [22]:
model.eval()
with torch.no_grad():
    max_tokens = 100

    token = torch.zeros((1, 1), dtype=torch.long, device=device)
    h_prev = torch.zeros((1, hidden_size), device=device)

    generated = []

    for _ in range(max_tokens):

        embds = model.embd(token)
        x_t = embds[:, 0, :]

        h_t = torch.tanh(model.W_xh(x_t) + model.W_hh(h_prev))
        y_t = model.W_hy(h_t)
        h_prev = h_t

        probs = F.softmax(y_t, dim=-1)
        token = torch.multinomial(probs,num_samples=1)

        generated.append(token.item())

    # decode generated ids
    output = tokenizer.decode(generated)

    print(output)

ility.
even beneats in the universal lie ritual of the exir creatt.
truth crom the nobility is the stretween structure the longly alone to underiphery.
the future fuaming.
the face of rnderly flow.
y shiftelle expand f
